In [2]:
from pathlib import Path
import os
import pandas as pd

# 1. Encontra a raiz do projeto procurando pela pasta 'data'
current_dir = Path(os.getcwd())
root_dir = current_dir if (current_dir / "data").exists() else current_dir.parent

raw_data_dir = root_dir / "data" / "raw"

print("A procurar na pasta:", raw_data_dir.resolve())
print("Ficheiros existentes em data/raw:", [f.name for f in raw_data_dir.glob("*")])

# 2. Seleciona o primeiro ficheiro Excel (.xlsx ou .xls) encontrado na pasta
excel_files = list(raw_data_dir.glob("*.xlsx")) + list(raw_data_dir.glob("*.xls"))

if not excel_files:
    raise FileNotFoundError(f"Nenhum ficheiro Excel encontrado em {raw_data_dir}. Verifica o nome do ficheiro guardado.")

file_path = excel_files[0]
print(f"A carregar: {file_path.name}")

# 3. Inspecionar folhas
xl = pd.ExcelFile(file_path)
print("Folhas disponíveis:", xl.sheet_names)

# 4. Pré-visualização das primeiras 10 linhas
df_preview = pd.read_excel(file_path, sheet_name=0, header=None, nrows=10)
df_preview

A procurar na pasta: C:\Users\joaop\Desktop\Smart Carb Assistant\data\raw
Ficheiros existentes em data/raw: ['.gitkeep', 'tca_insa.xlsx']
A carregar: tca_insa.xlsx
Folhas disponíveis: ['INSA - BDCA_v 7.1 - 2026', 'Componentes-Correspondência', 'Informação adicional']


,0,1,2,3,4,5,6,7,8,9,...,43,44,45,46,47,48,49,50,51,52
0,NaN,NaN,Grupo e subgrupos\n (Classificação FoodEx2 Nív...,NaN,NaN,Valores\npor 100 g de parte edível com exceção...,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Cod,Nome do alimento,Nível 1,Nível 2,Nível 3,Energia\n[kcal],Energia\n[kJ],Lípidos\n[g],Ácidos gordos saturados\n[g],Ácidos gordos monoinsaturados \n[g],...,Cinza \n[g],Sódio \n[mg],Potássio \n[mg],Cálcio \n[mg],Fósforo \n[mg],Magnésio \n[mg],Ferro \n[mg],Zinco \n[mg],Selénio \n[µg],Iodo \n[µg]
2,624,"Abacate, Hass",Frutos e produtos derivados de frutos,Fruta utilizada como fruta,"Frutos diversos com casca não comestível, grandes",176,726,17.4,4.2,10,...,0.75,15,330,4,36,21,0.3,0.3,NaN,NaN
3,625,Abóbora cristalizada,Produtos hortícolas e derivados,Produtos hortícolas transformados ou em conser...,Produtos hortícolas cristalizados ou conservad...,293,1240,0.2,0.1,0,...,0.15,27,22,28,2,3,0.4,0.1,NaN,NaN
4,579,Abóbora crua,Produtos hortícolas e derivados,Frutos de hortícolas,Frutos vegetais de cucurbitáceas,11,47,0.2,0.1,0,...,0.4,1,200,25,5,5,0.1,0.1,NaN,NaN
5,801,Abrótea cozida,"Peixes, mariscos, anfíbios, répteis e inverteb...",Peixe (músculo),Peixe de mar,79,334,0.1,0,0,...,2,360,360,13,270,36,0.2,0.6,NaN,22
6,800,Abrótea crua,"Peixes, mariscos, anfíbios, répteis e inverteb...",Peixe (músculo),Peixe de mar,70,296,0.1,0,0,...,1.1,63,360,11,230,28,0.2,0.5,NaN,NaN
7,1188,Açafrão,"Leguminosas, frutos de casca rija, sementes ol...",Especiarias,"Flores ou partes de flores, utilizadas como es...",353,1490,5.9,1.6,0.4,...,3,150,1720,110,250,50,11,1.1,NaN,NaN
8,1187,Açafrão-da-índia seco,"Leguminosas, frutos de casca rija, sementes ol...",Especiarias,Especiaria de de raízes e tubérculos,312,1300,7,2.9,0.6,...,7.08,31,2910,170,290,190,40,3.2,NaN,NaN
9,1900000023,Acelga crua,Produtos hortícolas e derivados,Hortícolas folhosos,Folhas do tipo espinafre,23,97,0.2,0,0,...,1.6,210,380,51,46,81,1.8,0.4,NaN,NaN


In [3]:
# 1. Carregar a folha correta com o cabeçalho na linha 1
sheet_name = 'INSA - BDCA_v 7.1 - 2026'
df_raw = pd.read_excel(file_path, sheet_name=sheet_name, header=1)

# 2. Limpar os nomes das colunas (remover quebras de linha e espaços extras)
df_raw.columns = [str(col).replace('\n', ' ').strip() for col in df_raw.columns]

# 3. Localizar dinamicamente as colunas essenciais
col_nome = [c for c in df_raw.columns if 'nome do alimento' in c.lower()][0]
col_hidratos = [c for c in df_raw.columns if 'hidratos de carbono' in c.lower()][0]
col_fibra = [c for c in df_raw.columns if 'fibra' in c.lower()][0]
col_energia = [c for c in df_raw.columns if 'energia' in c.lower() and 'kcal' in c.lower()][0]

print("Colunas selecionadas:")
print(f"- Nome: '{col_nome}'")
print(f"- Hidratos: '{col_hidratos}'")
print(f"- Fibra: '{col_fibra}'")
print(f"- Energia: '{col_energia}'")

# 4. Criar DataFrame padronizado
df_clean = df_raw[['Cod', col_nome, col_hidratos, col_fibra, col_energia]].copy()
df_clean.columns = ['id', 'nome', 'hidratos_100g', 'fibra_100g', 'kcal_100g']

# 5. Tratamento de tipos numéricos: converter 'Tr' (traços) e vazios em números reais
def sanitize_numeric(series):
    return (
        series.astype(str)
        .str.replace(',', '.', regex=False)
        .str.strip()
        .replace({'Tr': '0.0', 'nan': '0.0', '': '0.0', 'None': '0.0'})
        .astype(float)
    )

for col in ['hidratos_100g', 'fibra_100g', 'kcal_100g']:
    df_clean[col] = sanitize_numeric(df_clean[col])

# 6. Remover registos sem nome
df_clean = df_clean.dropna(subset=['nome'])

print(f"\nTotal de alimentos limpos: {len(df_clean)}")
df_clean.head(10)

Colunas selecionadas:
- Nome: 'Nome do alimento'
- Hidratos: 'Hidratos de carbono  [g]'
- Fibra: 'Fibra   [g]'
- Energia: 'Energia [kcal]'

Total de alimentos limpos: 1376


,id,nome,hidratos_100g,fibra_100g,kcal_100g
0,624,"Abacate, Hass",2.3,3.0,176.0
1,625,Abóbora cristalizada,72.4,0.7,293.0
2,579,Abóbora crua,1.7,0.7,11.0
3,801,Abrótea cozida,0.0,0.0,79.0
4,800,Abrótea crua,0.0,0.0,70.0
5,1188,Açafrão,61.5,3.9,353.0
6,1187,Açafrão-da-índia seco,44.1,22.7,312.0
7,1900000023,Acelga crua,2.7,1.6,23.0
8,510,"Achocolatado com alto teor de gordura, pó",70.6,1.0,428.0
9,509,"Achocolatado com baixo teor de gordura, pó",86.2,1.0,395.0


In [5]:
# Mapeamento inicial: Label de Visão -> Registo INSA + Pesos de Referência (g)
FOOD_MAPPING = {
    "rice": {
        "search_term": "Arroz branco cozido",
        "default_portion_g": 150.0,
        "portions": {"small": 100.0, "medium": 150.0, "large": 220.0},
        "uncertainty_pct": 0.15  # 15% margem de erro típica
    },
    "french_fries": {
        "search_term": "Batata frita",
        "default_portion_g": 100.0,
        "portions": {"small": 70.0, "medium": 100.0, "large": 160.0},
        "uncertainty_pct": 0.20
    },
    "boiled_potato": {
        "search_term": "Batata cozida sem pele",
        "default_portion_g": 130.0,
        "portions": {"small": 90.0, "medium": 130.0, "large": 200.0},
        "uncertainty_pct": 0.15
    },
    "pasta": {
        "search_term": "Massa cozida",
        "default_portion_g": 140.0,
        "portions": {"small": 90.0, "medium": 140.0, "large": 200.0},
        "uncertainty_pct": 0.15
    },
    "bread": {
        "search_term": "Pão de trigo",
        "default_portion_g": 50.0,
        "portions": {"small": 30.0, "medium": 50.0, "large": 80.0},
        "uncertainty_pct": 0.10
    },
    "chicken_breast": {
        "search_term": "Frango, peito, grelhado",
        "default_portion_g": 130.0,
        "portions": {"small": 90.0, "medium": 130.0, "large": 180.0},
        "uncertainty_pct": 0.10
    }
}

# Função de validação: cruzar com a base limpa do INSA
def validar_mapeamento(mapping, df):
    resolved = {}
    for key, cfg in mapping.items():
        term = cfg["search_term"]
        match = df[df['nome'].str.contains(term, case=False, na=False)]
        if not match.empty:
            item = match.iloc[0]
            resolved[key] = {
                "id_insa": item['id'],
                "nome_insa": item['nome'],
                "hidratos_100g": item['hidratos_100g'],
                "fibra_100g": item['fibra_100g'],
                "kcal_100g": item['kcal_100g'],
                "portions": cfg["portions"],
                "uncertainty_pct": cfg["uncertainty_pct"]
            }
        else:
            print(f"Aviso: '{term}' não encontrado diretamente na tabela!")
    return resolved

tabela_resolvida = validar_mapeamento(FOOD_MAPPING, df_clean)
pd.DataFrame.from_dict(tabela_resolvida, orient='index')[['nome_insa', 'hidratos_100g', 'fibra_100g']]

Aviso: 'Arroz branco cozido' não encontrado diretamente na tabela!
Aviso: 'Batata cozida sem pele' não encontrado diretamente na tabela!
Aviso: 'Massa cozida' não encontrado diretamente na tabela!
Aviso: 'Frango, peito, grelhado' não encontrado diretamente na tabela!


,nome_insa,hidratos_100g,fibra_100g
french_fries,Batata frita caseira (em palitos),27.6,2.4
bread,Pão de trigo,57.3,3.8


In [7]:
# Função auxiliar de pesquisa segura
def buscar_alimento(df, termo):
    resultado = df[df['nome'].str.contains(termo, case=False, na=False)]
    if resultado.empty:
        print(f"⚠️ Atenção: nenhum alimento encontrado para o termo '{termo}'")
        return None
    # Devolve a primeira correspondência
    return resultado.iloc[0]

# 1. Pesquisa mais ampla por termos simples
arroz = buscar_alimento(df_clean, "Arroz")
frango = buscar_alimento(df_clean, "Frango")
maca = buscar_alimento(df_clean, "Maçã")

# Confirmar quais foram os alimentos exatos selecionados da tabela
print("\n--- ALIMENTOS SELECIONADOS ---")
print("Arroz selecionado :", arroz['nome'] if arroz is not None else "Não encontrado")
print("Frango selecionado:", frango['nome'] if frango is not None else "Não encontrado")
print("Maçã selecionada  :", maca['nome'] if maca is not None else "Não encontrado")

# 2. Montar o prato apenas com os itens encontrados
prato_exemplo = []
if arroz is not None:
    prato_exemplo.append({"item": arroz, "peso_estimado_g": 150.0, "margem": 0.15})
if frango is not None:
    prato_exemplo.append({"item": frango, "peso_estimado_g": 130.0, "margem": 0.10})
if maca is not None:
    prato_exemplo.append({"item": maca, "peso_estimado_g": 120.0, "margem": 0.10})

# 3. Função de cálculo
def calcular_totais_prato(itens):
    total_esperado = 0.0
    total_min = 0.0
    total_max = 0.0

    print("\n--- DETALHE POR ALIMENTO ---")
    for entrada in itens:
        alimento = entrada["item"]
        peso = entrada["peso_estimado_g"]
        margem = entrada["margem"]
        
        hc_100g = alimento['hidratos_100g']
        
        hc_esperado = (peso * hc_100g) / 100
        hc_min = hc_esperado * (1.0 - margem)
        hc_max = hc_esperado * (1.0 + margem)
        
        total_esperado += hc_esperado
        total_min += hc_min
        total_max += hc_max
        
        print(f"• {alimento['nome'][:35]} (~{peso}g):")
        print(f"   HC: {hc_esperado:.1f}g (intervalo: {hc_min:.1f}g a {hc_max:.1f}g)")

    print("\n--- ESTIMATIVA TOTAL DA REFEIÇÃO ---")
    print(f"Hidratos Esperados : {total_esperado:.1f} g")
    print(f"Intervalo Seguro   : [{total_min:.1f} g  -  {total_max:.1f} g]")

# 4. Executar
calcular_totais_prato(prato_exemplo)


--- ALIMENTOS SELECIONADOS ---
Arroz selecionado : Arroz à valenciana
Frango selecionado: Arroz de frango
Maçã selecionada  : Doce de maçã

--- DETALHE POR ALIMENTO ---
• Arroz à valenciana (~150.0g):
   HC: 11.6g (intervalo: 9.8g a 13.3g)
• Arroz de frango (~130.0g):
   HC: 32.8g (intervalo: 29.5g a 36.0g)
• Doce de maçã (~120.0g):
   HC: 70.8g (intervalo: 63.7g a 77.9g)

--- ESTIMATIVA TOTAL DA REFEIÇÃO ---
Hidratos Esperados : 115.1 g
Intervalo Seguro   : [103.0 g  -  127.2 g]


In [8]:
# Pesquisar termos mais específicos para ver os nomes exatos no INSA
print("Opções de Arroz simples:")
print(df_clean[df_clean['nome'].str.contains('Arroz', case=False, na=False)][['id', 'nome', 'hidratos_100g']].head(5))

print("\nOpções de Frango simples:")
print(df_clean[df_clean['nome'].str.contains('Frango', case=False, na=False)][['id', 'nome', 'hidratos_100g']].head(5))

print("\nOpções de Maçã fruta:")
print(df_clean[df_clean['nome'].str.contains('Maçã', case=False, na=False)][['id', 'nome', 'hidratos_100g']].head(5))

Opções de Arroz simples:
      id                           nome  hidratos_100g
67  1087             Arroz à valenciana            7.7
68   402               Arroz agulha cru           78.1
69   400  Arroz carolino branqueado cru           79.6
70   403           Arroz cozido simples           28.0
71  1058              Arroz de bacalhau           15.3

Opções de Frango simples:
       id                                          nome  hidratos_100g
77    958                               Arroz de frango           25.2
78   1098         Arroz de frango com feijão e chouriço            8.8
79   1097  Arroz de frango malandrinho à moda de Monção            9.3
240  1116                      Canja de frango com aipo            2.3
241  1117                     Canja de frango com massa            2.1

Opções de Maçã fruta:
           id                                               nome  \
422       664                                       Doce de maçã   
480       946  Farinha láctea maç

In [9]:
# Exportar df_clean para a pasta processed
caminho_parquet = root_dir / "data" / "processed" / "tca_clean.parquet"
df_clean.to_parquet(caminho_parquet, index=False)
print(f"Base guardada com sucesso em: {caminho_parquet}")

Base guardada com sucesso em: c:\Users\joaop\Desktop\Smart Carb Assistant\data\processed\tca_clean.parquet
